<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>


# ROS 2 Workspaces and Packages

The previous notebooks used a command-line publisher to investigate messages, discovery, and names. That publisher supplied ready-made behavior: send a specified message at a specified rate on a specific topic.

A custom application needs its own source code that can be arbitrarily complex. How is that code organized, which libraries does it require, and how does ROS 2 know which function to run?

This notebook supplies a small Python publisher and subscriber built in a standard package rather than as a one-liner on the terminal. Inspecting their package will connect the source files to the information needed to build and run them. By the end of this notebook, the files responsible for application behavior, dependencies, and executable registration will be identifiable.

## Know the exercise workspace

As in previous notebooks, the commands proposed below run in a terminal inside the dedicated environment described in the [README](../README.md#prerequisites), starting from the LX checkout directory. Open the terminal and source ROS 2:

```bash
source /opt/ros/${ROS2_DISTRO}/setup.bash
```

(Successful sourcing normally returns to the prompt without output.)

A **workspace** groups packages that can be built together. The prepared environment records its workspace location in `COLCON_WS`:

```bash
printf 'Workspace: %s\n' "$COLCON_WS"
pwd
```

In the browser editor, these commands show the workspace and the current LX checkout:

```text
Workspace: /code
/code/src/lx-dd-middleware-ros2
```

The two paths serve different purposes. `/code` is the workspace root; the LX checkout contains this lesson and its supplied source files. The following commands use paths relative to that checkout, where `README.md`, `notebooks`, and `packages` are located. The alternate runtime described in the README mounts the checkout at a different location, but these relative commands remain valid when run from the LX checkout directory.

`COLCON_WS` is a convenience supplied by the exercise environment. Referring to it allows later workspace commands to use the configured location instead of assuming that every workspace is named `/code`.


## Find the supplied package

The source files are present, but can the build tool recognize them as a ROS 2 package?

**colcon** is the tool used to discover and build packages. Its `list` command inspects source packages without building them. From the LX checkout:

```bash
colcon list --base-paths packages
```

The supplied package appears as:

```text
ros2_pubsub	packages/ros2_pubsub	(ros.ament_python)
```

The three entries identify the package name, its source directory, and its package type. `ament_python` identifies the Python packaging approach used by this ROS 2 package.

`--base-paths packages` starts the search in this checkout’s `packages` directory, keeping observations focused on the source used in the exercise.

Finding the package establishes that `colcon` recognizes its source, but it does not mean the package was built, that its executables are available to `ros2 run`, or that it is configured correctly.

## Workspaces and packages

A package brings together application code and the information needed to install and use it. The files surrounding the Python code reveal those different responsibilities.

```bash
ls packages/ros2_pubsub
```

The listing identifies the top-level entries; the following structural overview also shows their relevant contents:

<figure id="figure-1" class="lx-figure" style="margin:1.5em auto; text-align:center;">
  <pre style="display:inline-block; margin:0; text-align:left;">
    packages/ros2_pubsub/
    |-- package.xml
    |-- resource/
    |   `-- ros2_pubsub
    |-- setup.cfg
    |-- setup.py
    `-- ros2_pubsub/
        |-- __init__.py
        |-- my_publisher.py
        `-- my_subscriber.py
  </pre>
  <figcaption style="font-size:0.9em; margin-top:0.6em; text-align:center;">
    Figure 1: Directory structure of the ros2_pubsub ament_python package.
  </figcaption>
</figure>

The inner `ros2_pubsub` directory contains the Python code that starts the nodes and implements their application behavior:

```bash
ls packages/ros2_pubsub/ros2_pubsub
```

Its source files are:

```text
__init__.py
my_publisher.py
my_subscriber.py
```

The repeated directory name reflects two layers: the outer directory holds the ROS 2 package, while the inner directory is the importable Python package. Later notebooks examine the publisher and subscriber implementations.

Before running either program, the package must describe what it needs and how its code should be installed.

### What does the code depend on?

The package manifest, `package.xml`, declares its identity and ROS dependencies:

```bash
cat packages/ros2_pubsub/package.xml
```

Among the package metadata, the following lines identify its name and dependencies:

```xml
<name>ros2_pubsub</name>
```

```xml
<buildtool_depend>ament_python</buildtool_depend>

<depend>rclpy</depend>
<depend>std_msgs</depend>
```

**Note that declaring a dependency records a requirement; it does not itself install the required software.**

These are excerpts from the file. `rclpy` provides the Python ROS 2 client library, while `std_msgs` supplies the message definitions used by this example, including the familiar `String` message.

The manifest also identifies the build type, which helps explain the package type reported by `colcon list`. 

```xml
<export>
  <build_type>ament_python</build_type>
</export>
```


### How does an executable name reach the Python code?

A source filename alone does not register a ROS 2 executable. The package’s installation configuration provides that connection:

```bash
cat packages/ros2_pubsub/setup.py
```

Near the end of the file, the `console_scripts` entries associate executable names with Python functions:

```python
entry_points={
    "console_scripts": [
        "my_publisher = ros2_pubsub.my_publisher:main",
        "my_subscriber = ros2_pubsub.my_subscriber:main",
    ],
},
```

The publisher entry can be read from left to right:

<table id="table-1" class="lx-table" style="margin:1.5em auto; text-align:left;">
  <caption style="caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center;">Table 1: Parts of a Python executable entry point.</caption>
  <thead>
    <tr>
      <th>Part</th>
      <th>Meaning</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>my_publisher</code></td>
      <td>Executable name to register during installation</td>
    </tr>
    <tr>
      <td><code>ros2_pubsub.my_publisher</code></td>
      <td>Python module containing the implementation</td>
    </tr>
    <tr>
      <td><code>main</code></td>
      <td>Function called when that executable starts</td>
    </tr>
  </tbody>
</table>

The subscriber entry follows the same pattern. Its executable will call `main` in `ros2_pubsub/my_subscriber.py`.

An executable name identifies something that can be launched. A node name identifies a participant in the running graph. Registering an executable does not, by itself, determine the node name used by its implementation.


### Where will the executable be installed?

The function mapping explains what should run, but ROS 2 also needs to find the installed executable. This information is contained in:

```bash
cat packages/ros2_pubsub/setup.cfg
```

The configuration shows:

```ini
[develop]
script_dir=$base/lib/ros2_pubsub
[install]
install_scripts=$base/lib/ros2_pubsub
```

These settings direct executable installation into the package-specific `lib/ros2_pubsub` directory beneath the installation base, where `ros2 run` expects to find the package’s executables. `$base` is an installation placeholder in this configuration file.

The remaining `resource/ros2_pubsub` marker is installed by `setup.py` into the ament package index, helping ROS 2 tools locate the installed package.

Together, these files connect the source code to an installed ROS 2 package:

<table id="table-2" class="lx-table" style="margin:1.5em auto; text-align:left;">
  <caption style="caption-side:top; font-size:0.9em; margin-bottom:0.6em; text-align:center;">Table 2: Responsibilities of the package files.</caption>
  <thead>
    <tr>
      <th>File or directory</th>
      <th>Responsibility</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>ros2_pubsub/*.py</code></td>
      <td>Implement the application behavior</td>
    </tr>
    <tr>
      <td><code>package.xml</code></td>
      <td>Declare package metadata, ROS dependencies, and build type</td>
    </tr>
    <tr>
      <td><code>setup.py</code></td>
      <td>Configure Python installation and executable entry points</td>
    </tr>
    <tr>
      <td><code>setup.cfg</code></td>
      <td>Select the executable installation directories</td>
    </tr>
    <tr>
      <td><code>resource/ros2_pubsub</code></td>
      <td>Provide the package-index marker</td>
    </tr>
  </tbody>
</table>

## Identify the right file before making a change

Each file has its own purpose, so edits should be made in the appropriate places. Suppose the publisher should send different text; that change belongs in its Python implementation. If the package needs an additional ROS dependency though, the manifest should describe it. If an executable should call a different function, its entry-point registration in `setup.py` must change.

These are different kinds of change, even though they belong to the same package. Identifying the responsible file helps avoid changing installation settings when the intended change concerns application behavior.

The supplied package already contains the structure needed for this LX. Creating a new package is a separate workflow, described in the [ROS 2 package-creation tutorial](https://docs.ros.org/en/jazzy/Tutorials/Beginner-Client-Libraries/Creating-Your-First-ROS2-Package.html).

Build tools also produce generated directories such as `build`, `install`, and `log`, which contain build products and records; application changes belong in the source files.

## Further reading

The [ROS 2 Jazzy package tutorial](https://docs.ros.org/en/jazzy/Tutorials/Beginner-Client-Libraries/Creating-Your-First-ROS2-Package.html) covers creating a package, its required files, and the steps from source to an installed executable.

## Checkpoint

The next cell contains the interactive self-check. A response must be entered or selected before the corresponding answer can be revealed. 

## Summary and next steps

The source inspection located `ros2_pubsub`, identified its dependencies, and traced executable names to Python functions. The surrounding configuration describes how those files become an installed package.

Source discovery is only the first step. [Notebook 6: Build and Discover a ROS 2 Package](./6-build-and-discover-a-ros-2-package.ipynb) builds the package and checks that ROS 2 can find its installed executables.

In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
